# Customer Churn Prediction\n\nIndustrial Practice ML Project\n\n**Student:** Amanzholova Dilnaz  \n**Program:** Mathematical and Computational Sciences  \n**Project:** Predictive Analytics System for Customer Churn Risk Assessment\n

## Objective\n\nThe objective of this notebook is to build a reproducible machine learning pipeline for customer churn prediction. The workflow includes data cleaning, exploratory analysis, preprocessing, model training, and evaluation.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import classification_report, roc_auc_score\n\nROOT = Path('..')\nDATA_PATH = ROOT / 'data' / 'customer_churn.csv'\nFIGURES_DIR = ROOT / 'figures'\nFIGURES_DIR.mkdir(exist_ok=True)\n

In [ ]:
df = pd.read_csv(DATA_PATH)\ndf.head()\n

In [ ]:
df.info()\ndf.isnull().sum()\n

## Exploratory Data Analysis\n

In [ ]:
plt.figure(figsize=(7, 5))\nsns.countplot(data=df, x='Churn')\nplt.title('Customer Churn Distribution')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'churn_distribution.png', dpi=200)\nplt.show()\n

In [ ]:
numeric_df = df.select_dtypes(include='number')\nplt.figure(figsize=(8, 6))\nsns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')\nplt.title('Correlation Heatmap')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=200)\nplt.show()\n

## Preprocessing and Model Training\n

In [ ]:
df = df.drop_duplicates()\ndf['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')\n\nX = df.drop(columns=['Churn'])\ny = df['Churn'].map({'Yes': 1, 'No': 0})\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42, stratify=y\n)\n\nnumeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns\ncategorical_features = X_train.select_dtypes(include=['object', 'category']).columns\n\nnumeric_pipeline = Pipeline([\n    ('imputer', SimpleImputer(strategy='median')),\n    ('scaler', StandardScaler())\n])\n\ncategorical_pipeline = Pipeline([\n    ('imputer', SimpleImputer(strategy='most_frequent')),\n    ('encoder', OneHotEncoder(handle_unknown='ignore'))\n])\n\npreprocessor = ColumnTransformer([\n    ('num', numeric_pipeline, numeric_features),\n    ('cat', categorical_pipeline, categorical_features)\n])\n

In [ ]:
models = {\n    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),\n    'Random Forest': RandomForestClassifier(\n        n_estimators=200, max_depth=8, random_state=42, class_weight='balanced'\n    )\n}\n\nresults = []\ntrained = {}\n\nfor name, model in models.items():\n    pipe = Pipeline([\n        ('preprocessor', preprocessor),\n        ('classifier', model)\n    ])\n    pipe.fit(X_train, y_train)\n    trained[name] = pipe\n    y_pred = pipe.predict(X_test)\n    y_proba = pipe.predict_proba(X_test)[:, 1]\n    print('\n' + name)\n    print(classification_report(y_test, y_pred))\n    print('ROC-AUC:', roc_auc_score(y_test, y_proba))\n    results.append({\n        'Model': name,\n        'ROC-AUC': roc_auc_score(y_test, y_proba)\n    })\n\npd.DataFrame(results)\n

## Feature Importance\n

In [ ]:
rf_pipe = trained['Random Forest']\nfeature_names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()\nimportances = rf_pipe.named_steps['classifier'].feature_importances_\n\nimportance_df = (\n    pd.DataFrame({'feature': feature_names, 'importance': importances})\n    .sort_values('importance', ascending=False)\n    .head(15)\n)\n\nplt.figure(figsize=(10, 7))\nsns.barplot(data=importance_df, x='importance', y='feature')\nplt.title('Top 15 Feature Importances')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'feature_importance.png', dpi=200)\nplt.show()\n